# Phase 0.0 — Small Model Pre-flight

**Purpose:** Determine whether Qwen3.5-4B-Instruct is viable as the 5th model in the uncertainty benchmark.

A model passes if it can:
1. Produce schema-compliant JSON on both pipeline schemas (natively or with constraints)
2. Return extractable top-k logprobs per generation step
3. Run within Colab T4 memory limits

**Runtime:** ~10-15 min on Colab T4 free tier.

**Result cell is at the bottom** — fill it in after running all probes.

In [1]:
# Install dependencies
# outlines for constrained decoding; bitsandbytes not needed at 4B
!pip install -q transformers accelerate outlines torch
!pip install -q sentencepiece protobuf  # Qwen tokenizer deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 7.7 MB/s eta 0:00:00


In [2]:
# ── HuggingFace authentication ───────────────────────────────────────────────
# Required to download Qwen weights from HF Hub.
#
# Option A (recommended): add your token to Colab Secrets
#   → click the key icon in the left sidebar → New secret → name: HF_TOKEN
# Option B: paste token directly in the fallback below
#
# Get a token at: huggingface.co → Settings → Access Tokens (Read permission)

from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("Logged in via Colab Secrets.")
except Exception:
    HF_TOKEN = ""  # <-- paste your token here if not using Colab Secrets
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Logged in via token.")
    else:
        raise RuntimeError(
            "No HF token found. Add HF_TOKEN to Colab Secrets (key icon in sidebar) "
            "or paste it into the HF_TOKEN variable above."
        )

Logged in via Colab Secrets.


In [3]:
import json
import re
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM total: 15.6 GB


In [4]:
# ── Load model ──────────────────────────────────────────────────────────────
# torch_dtype=auto picks bfloat16 on capable GPUs, float16 otherwise
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

if DEVICE == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM used after load: {used:.1f} / {total:.1f} GB")

print("Model loaded.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

VRAM used after load: 8.0 / 15.6 GB
Model loaded.


## Pipeline Schemas

These are the exact JSON schemas the benchmark pipelines will require.

**Single-turn schema** — model sees sparse context, asks 1 CQ, gives preliminary answer:
```json
{"clarifying_question": "<str>", "preliminary_answer": "<str>", "confidence": <int 0-100>}
```

**Flex-turn schema** — model decides each turn whether to keep asking:
```json
{"ask_cq": <bool>, "clarifying_question": "<str or null>", "preliminary_answer": "<str>", "confidence": <int 0-100>}
```
`ask_cq=false` means the model is done asking and commits to its answer.

In [10]:
# ── Shared helpers ───────────────────────────────────────────────────────────
import json
import torch

SINGLE_TURN_SCHEMA = {
    "clarifying_question": "str",
    "preliminary_answer": "str",
    "confidence": "int 0-100"
}

FLEX_TURN_SCHEMA = {
    "ask_cq": "bool",
    "clarifying_question": "str or null",
    "preliminary_answer": "str",
    "confidence": "int 0-100"
}

SYSTEM_PROMPT = (
    "You are an experienced clinician. You will receive a brief patient "
    "presentation and a clinical question. Respond ONLY with a valid JSON object."
)

SINGLE_TURN_INSTRUCTION = """Patient: 45-year-old male with sudden chest pain.
Question: What is the most appropriate next step?
Options: A) ECG, B) Chest X-ray, C) CT scan, D) Blood cultures

Respond with exactly this JSON and nothing else:
{\"clarifying_question\": \"<one targeted question>\", \"preliminary_answer\": \"<A, B, C, or D>\", \"confidence\": <0-100>}이야."""

FLEX_TURN_INSTRUCTION = """Patient: 45-year-old male with sudden chest pain.
Question: What is the most appropriate next step?
Options: A) ECG, B) Chest X-ray, C) CT scan, D) Blood cultures

If you need more information before committing, set ask_cq to true and provide a question.
If you are ready to commit, set ask_cq to false and set clarifying_question to null.

Respond with exactly this JSON and nothing else:
{\"ask_cq\": <true or false>, \"clarifying_question\": \"<question or null>\", \"preliminary_answer\": \"<A, B, C, or D>\", \"confidence\": <0-100>}이야."""

def build_messages(system, user):
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def generate(messages, max_new_tokens=256, temperature=0.0,
             return_logprobs=False, num_return_sequences=1,
             enable_thinking=False):
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature if temperature > 0 else None,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
    )
    if return_logprobs:
        gen_kwargs["output_scores"] = True
        gen_kwargs["return_dict_in_generate"] = True

    with torch.no_grad():
        # Use ** to unpack dictionary of tensors (input_ids, attention_mask, etc.)
        out = model.generate(**input_ids, **gen_kwargs)

    if return_logprobs:
        sequences = out.sequences
        scores = out.scores
    else:
        sequences = out
        scores = None

    # input_ids is a BatchEncoding, access tensor via key
    prompt_len = input_ids["input_ids"].shape[1]

    if num_return_sequences > 1:
        texts = [tokenizer.decode(sequences[i][prompt_len:], skip_special_tokens=True) for i in range(num_return_sequences)]
        return texts, scores
    else:
        text = tokenizer.decode(sequences[0][prompt_len:], skip_special_tokens=True)
        return text, scores

import re as _re
def strip_thinking(text):
    stripped = _re.sub(r'<think>.*?</think>', '', text, flags=_re.DOTALL)
    return stripped.strip()

def try_parse_json(text):
    text = strip_thinking(text).strip()
    try:
        return json.loads(text), "direct"
    except json.JSONDecodeError:
        pass
    match = _re.search(r'\{[^{}]*\}', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group()), "extracted"
        except json.JSONDecodeError:
            pass
    return None, "failed"

RESULTS = {}
print("Helpers ready and fixed.")

Helpers ready and fixed.


In [12]:
msgs = build_messages(SYSTEM_PROMPT, SINGLE_TURN_INSTRUCTION)
raw_thinking, _ = generate(msgs, max_new_tokens=4000, temperature=0.0, enable_thinking=True)

has_think_tag = "<think>" in raw_thinking
think_block = ""
after_think = raw_thinking

if has_think_tag:
    match = _re.search(r'<think>(.*?)</think>(.*)', raw_thinking, _re.DOTALL)
    if match:
        think_block = match.group(1).strip()
        after_think = match.group(2).strip()

print(f"Contains <think> block: {has_think_tag}")
if has_think_tag:
    print(f"Thinking length (chars): {len(think_block)}")
    print(f"Thinking preview: {think_block[:200]}...")
    print(f"\nOutput after </think>: {after_think[:300]}")
else:
    print(f"Raw output: {raw_thinking[:300]}")

parsed, method = try_parse_json(raw_thinking)
print(f"\nJSON parseable after stripping think tags: {parsed is not None} (method={method})")

RESULTS["probe0_thinking_mode"] = {
    "has_think_tag": has_think_tag,
    "think_length_chars": len(think_block),
    "json_parseable_after_strip": parsed is not None,
}
print("\nAll remaining probes use enable_thinking=False.")

Contains <think> block: True
Thinking length (chars): 927
Thinking preview: Okay, let's tackle this. The patient is a 45-year-old male with sudden chest pain. The question is about the most appropriate next step. The options are ECG, chest X-ray, CT scan, or blood cultures.

...

Output after </think>: {"clarifying_question": "Is the chest pain associated with any other symptoms like shortness of breath, radiation, or palpitations?", "preliminary_answer": "A", "confidence": 90}

JSON parseable after stripping think tags: True (method=direct)

All remaining probes use enable_thinking=False.


## Probe 1 — Native JSON compliance: single-turn schema

In [13]:
N_TRIALS = 5
parse_results = []

for i in range(N_TRIALS):
    msgs = build_messages(SYSTEM_PROMPT, SINGLE_TURN_INSTRUCTION)
    raw, _ = generate(msgs, max_new_tokens=4000, temperature=0.0, enable_thinking=False)
    parsed, method = try_parse_json(raw)

    keys_ok = False
    conf_ok = False
    if parsed:
        keys_ok = all(k in parsed for k in ("clarifying_question", "preliminary_answer", "confidence"))
        conf_ok = isinstance(parsed.get("confidence"), int) and 0 <= parsed["confidence"] <= 100

    status = "PASS" if (parsed and keys_ok and conf_ok) else "FAIL"
    parse_results.append(status)
    print(f"  Trial {i+1}: {status} | parse={method}")
    if parsed:
        print(f"    cq: {str(parsed.get('clarifying_question',''))[:80]}")
        print(f"    ans={parsed.get('preliminary_answer')}  conf={parsed.get('confidence')}")
    else:
        print(f"    raw: {raw[:120]}")

p1_pass_rate = parse_results.count("PASS") / N_TRIALS
RESULTS["probe1_single_turn_native"] = {"pass_rate": p1_pass_rate, "detail": parse_results}
print(f"\nProbe 1 pass rate: {p1_pass_rate:.0%}")

  Trial 1: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85
  Trial 2: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85
  Trial 3: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85
  Trial 4: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85
  Trial 5: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85

Probe 1 pass rate: 100%


## Probe 2 — Native JSON compliance: flex-turn schema

In [14]:
parse_results2 = []

for i in range(N_TRIALS):
    msgs = build_messages(SYSTEM_PROMPT, FLEX_TURN_INSTRUCTION)
    raw, _ = generate(msgs, max_new_tokens=128, temperature=0.0, enable_thinking=False)
    parsed, method = try_parse_json(raw)

    keys_ok = False
    types_ok = False
    if parsed:
        keys_ok = all(k in parsed for k in ("ask_cq", "clarifying_question", "preliminary_answer", "confidence"))
        ask_cq = parsed.get("ask_cq")
        ask_cq_bool = isinstance(ask_cq, bool)
        cq_consistent = True
        if ask_cq_bool and not ask_cq:
            cq_consistent = parsed.get("clarifying_question") is None
        conf_ok = isinstance(parsed.get("confidence"), int) and 0 <= parsed["confidence"] <= 100
        types_ok = ask_cq_bool and cq_consistent and conf_ok

    status = "PASS" if (parsed and keys_ok and types_ok) else "FAIL"
    parse_results2.append(status)
    print(f"  Trial {i+1}: {status} | parse={method}")
    if parsed:
        print(f"    ask_cq={parsed.get('ask_cq')}  ans={parsed.get('preliminary_answer')}  conf={parsed.get('confidence')}")
        print(f"    cq: {str(parsed.get('clarifying_question',''))[:80]}")
    else:
        print(f"    raw: {raw[:120]}")

p2_pass_rate = parse_results2.count("PASS") / N_TRIALS
RESULTS["probe2_flex_turn_native"] = {"pass_rate": p2_pass_rate, "detail": parse_results2}
print(f"\nProbe 2 pass rate: {p2_pass_rate:.0%}")

  Trial 1: PASS | parse=direct
    ask_cq=False  ans=A  conf=90
    cq: None
  Trial 2: PASS | parse=direct
    ask_cq=False  ans=A  conf=90
    cq: None
  Trial 3: PASS | parse=direct
    ask_cq=False  ans=A  conf=90
    cq: None
  Trial 4: PASS | parse=direct
    ask_cq=False  ans=A  conf=90
    cq: None
  Trial 5: PASS | parse=direct
    ask_cq=False  ans=A  conf=90
    cq: None

Probe 2 pass rate: 100%


## Probe 3 — Constrained decoding (outlines)

Only needed if Probes 1–2 pass rate < 100%. Tests whether grammar-constrained decoding can fix compliance failures.

In [15]:
import outlines
from pydantic import BaseModel
from typing import Optional

class SingleTurnOutput(BaseModel):
    clarifying_question: str
    preliminary_answer: str
    confidence: int

class FlexTurnOutput(BaseModel):
    ask_cq: bool
    clarifying_question: Optional[str]
    preliminary_answer: str
    confidence: int

try:
    # outlines wraps the HF model
    outline_model = outlines.models.transformers(MODEL_ID, model_kwargs={"torch_dtype": "auto", "device_map": "auto"})

    # Single-turn constrained
    gen_single = outlines.generate.json(outline_model, SingleTurnOutput)
    msgs = build_messages(SYSTEM_PROMPT, SINGLE_TURN_INSTRUCTION)
    prompt_text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    result_single = gen_single(prompt_text)
    print("Constrained single-turn:", result_single)

    # Flex-turn constrained
    gen_flex = outlines.generate.json(outline_model, FlexTurnOutput)
    msgs2 = build_messages(SYSTEM_PROMPT, FLEX_TURN_INSTRUCTION)
    prompt_text2 = tokenizer.apply_chat_template(msgs2, add_generation_prompt=True, tokenize=False)
    result_flex = gen_flex(prompt_text2)
    print("Constrained flex-turn:", result_flex)

    RESULTS["probe3_constrained"] = {"status": "PASS", "single": str(result_single), "flex": str(result_flex)}
    print("\nProbe 3: PASS — outlines constrained decoding works")

except Exception as e:
    RESULTS["probe3_constrained"] = {"status": "FAIL", "error": str(e)}
    print(f"Probe 3: FAIL — {e}")
    print("Note: if outlines fails, JSONLogitsProcessor from transformers is the fallback (see Probe 3b below)")

Probe 3: FAIL — 'module' object is not callable
Note: if outlines fails, JSONLogitsProcessor from transformers is the fallback (see Probe 3b below)


In [16]:
# Probe 3b — fallback: manual JSON enforcement via prompt + regex extraction
# This is the lowest-cost fallback if outlines doesn't work.
# Run this regardless to confirm the retry ladder works.

STRICT_SINGLE_INSTRUCTION = """Patient: 45-year-old male with sudden chest pain radiating to left arm.
Question: What is the most appropriate next step?
Options: A) ECG, B) Chest X-ray, C) CT scan, D) Blood cultures

You MUST respond with ONLY the following JSON. Do NOT include any explanation or text outside the JSON.
{"clarifying_question": "...", "preliminary_answer": "A", "confidence": 85}"""

msgs = build_messages(SYSTEM_PROMPT, STRICT_SINGLE_INSTRUCTION)
raw, _ = generate(msgs, max_new_tokens=100, temperature=0.0)
parsed, method = try_parse_json(raw)
status = "PASS" if parsed and all(k in parsed for k in ("clarifying_question", "preliminary_answer", "confidence")) else "FAIL"

RESULTS["probe3b_strict_prompt"] = {"status": status, "parse_method": method}
print(f"Probe 3b (strict prompt + regex extraction): {status} via {method}")
print("Raw:", raw[:200])

Probe 3b (strict prompt + regex extraction): PASS via direct
Raw: {"clarifying_question": "Is the patient experiencing any other symptoms such as shortness of breath, sweating, or nausea?", "preliminary_answer": "A", "confidence": 85}


## Probe 4 — Logprob extraction

Critical for open-model UQ. We need `output_scores` to return a probability distribution over the vocabulary at each generation step.

In [17]:
msgs = build_messages(SYSTEM_PROMPT, SINGLE_TURN_INSTRUCTION)
raw, scores = generate(msgs, max_new_tokens=64, temperature=0.0, return_logprobs=True)

print(f"Generated text: {raw[:100]}")
print(f"Number of generation steps (tokens): {len(scores)}")

if scores and len(scores) > 0:
    # Each element of scores is a (1, vocab_size) tensor of raw logits
    step0_logits = scores[0][0]  # shape: (vocab_size,)
    step0_probs = torch.softmax(step0_logits, dim=-1)

    # Top-5 tokens at first generation step
    top_k = 5
    top_probs, top_ids = torch.topk(step0_probs, top_k)
    print(f"\nTop-{top_k} tokens at step 0:")
    for prob, tok_id in zip(top_probs.tolist(), top_ids.tolist()):
        tok = tokenizer.decode([tok_id])
        print(f"  '{tok}' (id={tok_id}) — p={prob:.4f}")

    # Per-token entropy across all steps
    entropies = []
    for step_logits in scores:
        probs = torch.softmax(step_logits[0], dim=-1)
        # Clip to avoid log(0)
        probs_clipped = torch.clamp(probs, min=1e-12)
        entropy = -torch.sum(probs_clipped * torch.log(probs_clipped)).item()
        entropies.append(entropy)

    print(f"\nPer-token entropy stats across {len(entropies)} steps:")
    print(f"  Mean:   {np.mean(entropies):.4f}")
    print(f"  Median: {np.median(entropies):.4f}")
    print(f"  Max:    {np.max(entropies):.4f}")
    print(f"  Min:    {np.min(entropies):.4f}")

    RESULTS["probe4_logprobs"] = {
        "status": "PASS",
        "n_steps": len(scores),
        "mean_entropy": float(np.mean(entropies)),
        "vocab_size": int(step0_logits.shape[0])
    }
    print("\nProbe 4: PASS — logprob extraction works")
else:
    RESULTS["probe4_logprobs"] = {"status": "FAIL", "reason": "scores is empty"}
    print("\nProbe 4: FAIL — scores is empty")

Generated text: {"clarifying_question": "Is the chest pain associated with any other symptoms such as shortness of b
Number of generation steps (tokens): 46

Top-5 tokens at step 0:
  '{"' (id=4913) — p=0.9046
  '{
' (id=515) — p=0.0953
  '{' (id=90) — p=0.0000
  '```' (id=73594) — p=0.0000
  '{'' (id=13608) — p=0.0000

Per-token entropy stats across 46 steps:
  Mean:   0.1331
  Median: 0.0000
  Max:    1.5317
  Min:    0.0000

Probe 4: PASS — logprob extraction works


## Probe 5 — Multi-sample generation (semantic entropy feasibility)

Semantic entropy requires N independent samples at T > 0. We check that the model produces meaningfully different outputs across samples and that generation is fast enough.

In [18]:
import time

N_SAMPLES = 3
T = 0.7

msgs = build_messages(SYSTEM_PROMPT, SINGLE_TURN_INSTRUCTION)

t0 = time.time()
texts, _ = generate(msgs, max_new_tokens=128, temperature=T, num_return_sequences=N_SAMPLES)
elapsed = time.time() - t0

print(f"{N_SAMPLES} samples generated in {elapsed:.1f}s")

parsed_samples = []
for i, raw in enumerate(texts):
    parsed, method = try_parse_json(raw)
    keys_ok = parsed and all(k in parsed for k in ("clarifying_question", "preliminary_answer", "confidence"))
    parsed_samples.append(parsed if keys_ok else None)
    status = "PASS" if keys_ok else "FAIL"
    print(f"  Sample {i+1}: {status} | parse={method}")
    if parsed:
        print(f"    cq: {str(parsed.get('clarifying_question',''))[:80]}")
        print(f"    ans={parsed.get('preliminary_answer')}  conf={parsed.get('confidence')}")

# Check diversity: are the CQs different across samples?
cqs = [p["clarifying_question"] for p in parsed_samples if p and p.get("clarifying_question")]
unique_cqs = len(set(cqs))
print(f"\nUnique CQs across {len(cqs)} parseable samples: {unique_cqs}")
diverse = unique_cqs > 1

RESULTS["probe5_multisample"] = {
    "status": "PASS" if diverse else "WARN_LOW_DIVERSITY",
    "n_samples": N_SAMPLES,
    "parse_rate": sum(1 for p in parsed_samples if p) / N_SAMPLES,
    "unique_cqs": unique_cqs,
    "elapsed_s": round(elapsed, 1)
}
print(f"Probe 5: {'PASS' if diverse else 'WARN — low diversity, semantic entropy may not be meaningful'}")

3 samples generated in 14.3s
  Sample 1: PASS | parse=direct
    cq: Is the chest pain associated with shortness of breath or other symptoms of a car
    ans=A  conf=90
  Sample 2: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85
  Sample 3: PASS | parse=direct
    cq: Is the chest pain associated with any other symptoms such as shortness of breath
    ans=A  conf=85

Unique CQs across 3 parseable samples: 3
Probe 5: PASS


## Probe 6 — Token budget: find max_new_tokens needed

Check at what `max_new_tokens` the model produces complete JSON without truncation.

In [19]:
msgs = build_messages(SYSTEM_PROMPT, FLEX_TURN_INSTRUCTION)  # flex has more fields → longer

for budget in [64, 128, 256]:
    raw, _ = generate(msgs, max_new_tokens=budget, temperature=0.0)
    parsed, method = try_parse_json(raw)
    n_tokens = len(tokenizer.encode(raw))
    truncated = not raw.strip().endswith("}")
    status = "TRUNCATED" if truncated else ("PASS" if parsed else "PARSE_FAIL")
    print(f"  budget={budget:3d}: {status} | tokens_used={n_tokens} | parse={method}")
    if parsed:
        print(f"    ask_cq={parsed.get('ask_cq')}  conf={parsed.get('confidence')}")

RESULTS["probe6_token_budget"] = "see output above"

  budget= 64: PASS | tokens_used=30 | parse=direct
    ask_cq=False  conf=90
  budget=128: PASS | tokens_used=30 | parse=direct
    ask_cq=False  conf=90
  budget=256: PASS | tokens_used=30 | parse=direct
    ask_cq=False  conf=90


---
## Findings Summary

Run this cell after all probes complete. Fill in the **Decision** field manually.

In [20]:
print("=" * 60)
print("PHASE 0.0 — PRE-FLIGHT RESULTS")
print("=" * 60)

p0 = RESULTS.get("probe0_thinking_mode", {})
print(f"\n[INFO] Thinking mode (enable_thinking=True):")
print(f"  Has <think> block:          {p0.get('has_think_tag', 'not run')}")
print(f"  Think block length (chars): {p0.get('think_length_chars', 'N/A')}")
print(f"  JSON parseable after strip: {p0.get('json_parseable_after_strip', 'N/A')}")

checks = {
    "JSON compliance (single-turn, thinking off)": RESULTS.get("probe1_single_turn_native", {}).get("pass_rate", 0) >= 0.8,
    "JSON compliance (flex-turn, thinking off)":   RESULTS.get("probe2_flex_turn_native", {}).get("pass_rate", 0) >= 0.8,
    "Constrained decoding available":              RESULTS.get("probe3_constrained", {}).get("status") == "PASS",
    "Logprob extraction works":                    RESULTS.get("probe4_logprobs", {}).get("status") == "PASS",
    "Multi-sample diversity (T=0.7)":              RESULTS.get("probe5_multisample", {}).get("unique_cqs", 0) > 1,
    "Runs on T4 (no OOM)":                         DEVICE == "cuda",
}

print()
for check, passed in checks.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")

json_ok = checks["JSON compliance (single-turn, thinking off)"] or checks["Constrained decoding available"]
logprob_ok = checks["Logprob extraction works"]
memory_ok = checks["Runs on T4 (no OOM)"]

verdict = "IN" if (json_ok and logprob_ok and memory_ok) else "OUT"

print()
print(f"JSON compliance (native OR constrained): {'OK' if json_ok else 'FAIL'}")
print(f"Logprob extraction:                      {'OK' if logprob_ok else 'FAIL'}")
print(f"Memory (T4):                             {'OK' if memory_ok else 'FAIL'}")
print()
print(f">>> QWEN3.5-4B VERDICT: {verdict} <<<")
print()

if verdict == "IN":
    print("Qwen3.5-4B enters the main model matrix.")
    p4 = RESULTS.get("probe4_logprobs", {})
    p5 = RESULTS.get("probe5_multisample", {})
    print("Recommended settings:")
    print(f"  enable_thinking:    False (suppress think blocks for structured output)")
    print(f"  max_new_tokens:     128 (or 256 if truncation seen in Probe 6)")
    print(f"  temperature:        0.0 for experiments, 0.7 for semantic entropy sampling")
    print(f"  constrained:        {'yes (outlines)' if checks['Constrained decoding available'] else 'no (native + regex fallback)'}")
    print(f"  logprob vocab_size: {p4.get('vocab_size', 'N/A')}")
    print(f"  mean_entropy T=0:   {p4.get('mean_entropy', 'N/A')}")
    print(f"  semantic_entropy_feasible: {p5.get('unique_cqs', 0) > 1}")
else:
    print("Qwen3.5-4B does NOT enter the main model matrix.")
    if not json_ok:    print("  - Cannot produce schema-compliant JSON")
    if not logprob_ok: print("  - Logprob extraction failed")
    if not memory_ok:  print("  - OOM on T4")

print()
print("Update CHECKLIST.md: Phase 0.0 complete, Decision recorded.")

PHASE 0.0 — PRE-FLIGHT RESULTS

[INFO] Thinking mode (enable_thinking=True):
  Has <think> block:          True
  Think block length (chars): 927
  JSON parseable after strip: True

  [PASS] JSON compliance (single-turn, thinking off)
  [PASS] JSON compliance (flex-turn, thinking off)
  [FAIL] Constrained decoding available
  [PASS] Logprob extraction works
  [PASS] Multi-sample diversity (T=0.7)
  [PASS] Runs on T4 (no OOM)

JSON compliance (native OR constrained): OK
Logprob extraction:                      OK
Memory (T4):                             OK

>>> QWEN3.5-4B VERDICT: IN <<<

Qwen3.5-4B enters the main model matrix.
Recommended settings:
  enable_thinking:    False (suppress think blocks for structured output)
  max_new_tokens:     128 (or 256 if truncation seen in Probe 6)
  temperature:        0.0 for experiments, 0.7 for semantic entropy sampling
  constrained:        no (native + regex fallback)
  logprob vocab_size: 151936
  mean_entropy T=0:   0.133057623606587
  sema

In [21]:
# Save full probe results to JSON for reference
import json as _json
with open("preflight_results.json", "w") as f:
    _json.dump(RESULTS, f, indent=2)
print("Saved preflight_results.json")

Saved preflight_results.json
